# Indianapolis e-scooter data preprocessing and minimum fleet size

This notebook is a cleaned version of the code used for the Indianapolis case study. It keeps the main steps only:

1. preprocess the original Purdue e-scooter trip data and generate `clean_data_reproduced.csv`;
2. match trip origins and destinations to census tracts and generate `OD_data_reproduced.csv`;
3. calculate the minimum fleet size using the trip-chain algorithm.

The original dataset can be downloaded from Purdue University: DOI `10.4231/3FT5-MJ18`.

This notebook is intended for teaching/reproducibility. The output file names use `_reproduced` to avoid overwriting the original project files.

## Data preparation

Before running this notebook, please download the Purdue file `purr_scooter_data.csv` and put it in the `data` folder:

`data/purr_scooter_data.csv`

The census tract shapefile should be placed here:

`data/tl_2019_18_tract/tl_2019_18_tract.shp`

In [35]:
import pandas as pd
import numpy as np
import geopandas as gpd
import pytz
import math
import time
import warnings

warnings.filterwarnings('ignore')
warnings.simplefilter('ignore')

## 1. Read and clean the original trip data

This part follows the logic of the original preprocessing notebook. The cleaned file is saved as `clean_data_reproduced.csv`.

In [37]:
# read the original trip data
data = pd.read_csv('data\purr_scooter_data.csv')
data.head()

,trip_id,scooter_id,start_time_utc,end_time_utc,start_lat,start_lon,end_lat,end_lon,distance_miles
0,1bcd169434884f290bfb5dbaad15fc2fcff6e76f69fa0e...,89118a5b9c8cf80e5b0f1799e61c140eed404bef159f1e...,2018-09-04T09:33:00Z,2018-09-04T10:27:00Z,39.766650,-86.160210,39.758495,-86.185028,6.69
1,0c3cb040ecc4d11d8e0bd1b44ff6b75777a6b7338ea629...,872ccf3550bc1d1ed5542ee004958a21d5f6d3039b0813...,2018-09-04T09:57:00Z,2018-09-04T09:59:00Z,39.767559,-86.159926,39.768615,-86.158798,0.35
2,25089f1a3f3d31b334104d2d8a9aabab2ca111de60bb77...,68b8d9d5dc482c8c9dd2009606af6eea0fad33e068f8c5...,2018-09-04T10:12:00Z,2018-09-04T10:15:00Z,39.777007,-86.146346,39.775085,-86.146225,0.35
3,8101dedc142b739f19c0c91d28b94b2ae7f2dc08f73c34...,fa979aa810d7094db412b497a091ee8d77815ddffbac0d...,2018-09-04T10:21:00Z,2018-09-04T10:27:00Z,39.780555,-86.170685,39.778378,-86.178475,0.58
4,2c308891a945fc93e2a6a66177ed463bfc822895090fc1...,3f097939c60c50fcbd35604b34edfc2dfe1f09ecbb6cfb...,2018-09-04T10:42:00Z,2018-09-04T10:50:00Z,39.769585,-86.156090,39.764035,-86.150330,0.97


In [38]:
# remove missing values
clean_data = data.copy()
clean_data = clean_data.dropna(how='any')
clean_data = clean_data.replace(to_replace='None', value=np.nan).dropna(how='any')
clean_data.head()

,trip_id,scooter_id,start_time_utc,end_time_utc,start_lat,start_lon,end_lat,end_lon,distance_miles
0,1bcd169434884f290bfb5dbaad15fc2fcff6e76f69fa0e...,89118a5b9c8cf80e5b0f1799e61c140eed404bef159f1e...,2018-09-04T09:33:00Z,2018-09-04T10:27:00Z,39.766650,-86.160210,39.758495,-86.185028,6.69
1,0c3cb040ecc4d11d8e0bd1b44ff6b75777a6b7338ea629...,872ccf3550bc1d1ed5542ee004958a21d5f6d3039b0813...,2018-09-04T09:57:00Z,2018-09-04T09:59:00Z,39.767559,-86.159926,39.768615,-86.158798,0.35
2,25089f1a3f3d31b334104d2d8a9aabab2ca111de60bb77...,68b8d9d5dc482c8c9dd2009606af6eea0fad33e068f8c5...,2018-09-04T10:12:00Z,2018-09-04T10:15:00Z,39.777007,-86.146346,39.775085,-86.146225,0.35
3,8101dedc142b739f19c0c91d28b94b2ae7f2dc08f73c34...,fa979aa810d7094db412b497a091ee8d77815ddffbac0d...,2018-09-04T10:21:00Z,2018-09-04T10:27:00Z,39.780555,-86.170685,39.778378,-86.178475,0.58
4,2c308891a945fc93e2a6a66177ed463bfc822895090fc1...,3f097939c60c50fcbd35604b34edfc2dfe1f09ecbb6cfb...,2018-09-04T10:42:00Z,2018-09-04T10:50:00Z,39.769585,-86.156090,39.764035,-86.150330,0.97


In [39]:
# convert UTC time to Indianapolis local time
indianapolis_tz = pytz.timezone('America/Indiana/Indianapolis')

clean_data['start_time'] = pd.to_datetime(clean_data['start_time_utc'])
clean_data['start_time'] = clean_data['start_time'].dt.tz_localize(None).dt.tz_localize('UTC').dt.tz_convert(indianapolis_tz)

clean_data['end_time'] = pd.to_datetime(clean_data['end_time_utc'])
clean_data['end_time'] = clean_data['end_time'].dt.tz_localize(None).dt.tz_localize('UTC').dt.tz_convert(indianapolis_tz)

clean_data = clean_data[['trip_id','scooter_id','start_time','end_time','start_lat','start_lon','end_lat','end_lon','distance_miles']]
clean_data.head()

,trip_id,scooter_id,start_time,end_time,start_lat,start_lon,end_lat,end_lon,distance_miles
0,1bcd169434884f290bfb5dbaad15fc2fcff6e76f69fa0e...,89118a5b9c8cf80e5b0f1799e61c140eed404bef159f1e...,2018-09-04 05:33:00-04:00,2018-09-04 06:27:00-04:00,39.766650,-86.160210,39.758495,-86.185028,6.69
1,0c3cb040ecc4d11d8e0bd1b44ff6b75777a6b7338ea629...,872ccf3550bc1d1ed5542ee004958a21d5f6d3039b0813...,2018-09-04 05:57:00-04:00,2018-09-04 05:59:00-04:00,39.767559,-86.159926,39.768615,-86.158798,0.35
2,25089f1a3f3d31b334104d2d8a9aabab2ca111de60bb77...,68b8d9d5dc482c8c9dd2009606af6eea0fad33e068f8c5...,2018-09-04 06:12:00-04:00,2018-09-04 06:15:00-04:00,39.777007,-86.146346,39.775085,-86.146225,0.35
3,8101dedc142b739f19c0c91d28b94b2ae7f2dc08f73c34...,fa979aa810d7094db412b497a091ee8d77815ddffbac0d...,2018-09-04 06:21:00-04:00,2018-09-04 06:27:00-04:00,39.780555,-86.170685,39.778378,-86.178475,0.58
4,2c308891a945fc93e2a6a66177ed463bfc822895090fc1...,3f097939c60c50fcbd35604b34edfc2dfe1f09ecbb6cfb...,2018-09-04 06:42:00-04:00,2018-09-04 06:50:00-04:00,39.769585,-86.156090,39.764035,-86.150330,0.97


In [40]:
# get date information
clean_data['year'] = pd.to_datetime(clean_data['start_time']).dt.year
clean_data['month'] = pd.to_datetime(clean_data['start_time']).dt.month
clean_data['hour'] = pd.to_datetime(clean_data['start_time']).dt.hour
clean_data['day'] = pd.to_datetime(clean_data['start_time']).dt.day
clean_data['week'] = pd.to_datetime(clean_data['start_time']).dt.dayofweek
clean_data.head()

,trip_id,scooter_id,start_time,end_time,start_lat,start_lon,end_lat,end_lon,distance_miles,year,month,hour,day,week
0,1bcd169434884f290bfb5dbaad15fc2fcff6e76f69fa0e...,89118a5b9c8cf80e5b0f1799e61c140eed404bef159f1e...,2018-09-04 05:33:00-04:00,2018-09-04 06:27:00-04:00,39.766650,-86.160210,39.758495,-86.185028,6.69,2018,9,5,4,1
1,0c3cb040ecc4d11d8e0bd1b44ff6b75777a6b7338ea629...,872ccf3550bc1d1ed5542ee004958a21d5f6d3039b0813...,2018-09-04 05:57:00-04:00,2018-09-04 05:59:00-04:00,39.767559,-86.159926,39.768615,-86.158798,0.35,2018,9,5,4,1
2,25089f1a3f3d31b334104d2d8a9aabab2ca111de60bb77...,68b8d9d5dc482c8c9dd2009606af6eea0fad33e068f8c5...,2018-09-04 06:12:00-04:00,2018-09-04 06:15:00-04:00,39.777007,-86.146346,39.775085,-86.146225,0.35,2018,9,6,4,1
3,8101dedc142b739f19c0c91d28b94b2ae7f2dc08f73c34...,fa979aa810d7094db412b497a091ee8d77815ddffbac0d...,2018-09-04 06:21:00-04:00,2018-09-04 06:27:00-04:00,39.780555,-86.170685,39.778378,-86.178475,0.58,2018,9,6,4,1
4,2c308891a945fc93e2a6a66177ed463bfc822895090fc1...,3f097939c60c50fcbd35604b34edfc2dfe1f09ecbb6cfb...,2018-09-04 06:42:00-04:00,2018-09-04 06:50:00-04:00,39.769585,-86.156090,39.764035,-86.150330,0.97,2018,9,6,4,1


In [41]:
# save cleaned data
# The original project used data/clean_data.csv.
# Here I use a reproduced file name to avoid overwriting the original file.
clean_data.to_csv('data/clean_data_reproduced.csv')

## 2. Generate OD_data_reproduced.csv

This part follows the original OD generation logic from the project:

`clean_data.csv -> July 2019 trips -> spatial join with census tracts -> OD_data.csv`

Here the output is saved as `OD_data_reproduced.csv`. It has the same structure as the original `OD_data.csv`.

In [42]:
# read cleaned trip data
trip_data = pd.read_csv('data/clean_data_reproduced.csv')
trip_data = trip_data.replace(to_replace='None', value=np.nan).dropna(how='any')
trip_data = trip_data.drop(['Unnamed: 0'], axis=1)

# The original OD_data.csv used for the minimum fleet size analysis is July 2019.
trip_data = trip_data[(trip_data['month'] == 7) & (trip_data['year'] == 2019)]
trip_data = trip_data.sort_values(by=['month','day'])
trip_data.head()

,trip_id,scooter_id,start_time,end_time,start_lat,start_lon,end_lat,end_lon,distance_miles,year,month,hour,day,week
1015755,f039fe4f9524bc049e7e6fca2d6e7be5757b2b2414c748...,cddc4d02e40050512ca2d67db69857f35e3854b3ab5f14...,2019-07-01 00:00:19-04:00,2019-07-01 00:03:40-04:00,39.77402,-86.16501,39.77516,-86.16492,0.396,2019,7,0,1,0
1015756,1847fe5c4f6a60177b2249518811a31f1616a03da405c1...,233c5f053e434c87fd3fe721b6b44117d4fb27397f09ab...,2019-07-01 00:02:33-04:00,2019-07-01 00:10:24-04:00,39.77472,-86.14957,39.76879,-86.15260,0.473,2019,7,0,1,0
1015757,0cc4befe4ea796cc3958e4e082f928ebde218d66b314c7...,a9110f1d9b1df5975bc9e39d6fbd8757bea15023464a6e...,2019-07-01 00:03:04-04:00,2019-07-01 00:47:50-04:00,39.76426,-86.15844,39.77409,-86.15520,2.687,2019,7,0,1,0
1015758,58172117412704ebd442eebb71a8f1452b6d5abe7dbb99...,0b6c1ea28a1545a8165d7500e9f3f0a4b7aa8ea23f4542...,2019-07-01 00:03:44-04:00,2019-07-01 00:16:25-04:00,39.75571,-86.15934,39.76567,-86.16683,0.994,2019,7,0,1,0
1015759,ff41bcabc7fc7b8d1101f2b4a7d3e73f3ef36c6c7bf3ee...,14b4f857e27f20427a54b1b1c7d496cd15b84dff84d4ad...,2019-07-01 00:03:46-04:00,2019-07-01 00:08:24-04:00,39.76425,-86.15730,39.76711,-86.16811,0.000,2019,7,0,1,0


In [43]:
# read census tract polygons
tract_2019 = gpd.read_file('data/tl_2019_18_tract/tl_2019_18_tract.shp')
tract = tract_2019[['TRACTCE','geometry']].drop_duplicates('geometry')
tract.columns = ['tract_name','geometry']
tract.head()

,tract_name,geometry
0,950400,"POLYGON ((-86.69580 40.18009, -86.69580 40.184..."
1,950100,"POLYGON ((-86.43202 40.25585, -86.43202 40.257..."
2,950200,"POLYGON ((-86.56824 40.41791, -86.56824 40.417..."
3,950500,"POLYGON ((-86.54333 40.28371, -86.54332 40.285..."
4,950300,"POLYGON ((-86.69522 40.33266, -86.69511 40.344..."


In [44]:
# create start points
start_point = trip_data[['trip_id','scooter_id','start_time','end_time','start_lon','start_lat','hour','day','month','distance_miles']]
start_point['geometry'] = gpd.points_from_xy(start_point['start_lon'], start_point['start_lat'])
start_point = gpd.GeoDataFrame(start_point, geometry=start_point['geometry'], crs='EPSG:4269')
start_point.head()

,trip_id,scooter_id,start_time,end_time,start_lon,start_lat,hour,day,month,distance_miles,geometry
1015755,f039fe4f9524bc049e7e6fca2d6e7be5757b2b2414c748...,cddc4d02e40050512ca2d67db69857f35e3854b3ab5f14...,2019-07-01 00:00:19-04:00,2019-07-01 00:03:40-04:00,-86.16501,39.77402,0,1,7,0.396,POINT (-86.16501 39.77402)
1015756,1847fe5c4f6a60177b2249518811a31f1616a03da405c1...,233c5f053e434c87fd3fe721b6b44117d4fb27397f09ab...,2019-07-01 00:02:33-04:00,2019-07-01 00:10:24-04:00,-86.14957,39.77472,0,1,7,0.473,POINT (-86.14957 39.77472)
1015757,0cc4befe4ea796cc3958e4e082f928ebde218d66b314c7...,a9110f1d9b1df5975bc9e39d6fbd8757bea15023464a6e...,2019-07-01 00:03:04-04:00,2019-07-01 00:47:50-04:00,-86.15844,39.76426,0,1,7,2.687,POINT (-86.15844 39.76426)
1015758,58172117412704ebd442eebb71a8f1452b6d5abe7dbb99...,0b6c1ea28a1545a8165d7500e9f3f0a4b7aa8ea23f4542...,2019-07-01 00:03:44-04:00,2019-07-01 00:16:25-04:00,-86.15934,39.75571,0,1,7,0.994,POINT (-86.15934 39.75571)
1015759,ff41bcabc7fc7b8d1101f2b4a7d3e73f3ef36c6c7bf3ee...,14b4f857e27f20427a54b1b1c7d496cd15b84dff84d4ad...,2019-07-01 00:03:46-04:00,2019-07-01 00:08:24-04:00,-86.15730,39.76425,0,1,7,0.000,POINT (-86.15730 39.76425)


In [45]:
# create end points
end_point = trip_data[['trip_id','scooter_id','start_time','end_time','end_lon','end_lat','hour','day','month','distance_miles']]
end_point['geometry'] = gpd.points_from_xy(end_point['end_lon'], end_point['end_lat'])
end_point = gpd.GeoDataFrame(end_point, geometry=end_point['geometry'], crs='EPSG:4269')
end_point.head()

,trip_id,scooter_id,start_time,end_time,end_lon,end_lat,hour,day,month,distance_miles,geometry
1015755,f039fe4f9524bc049e7e6fca2d6e7be5757b2b2414c748...,cddc4d02e40050512ca2d67db69857f35e3854b3ab5f14...,2019-07-01 00:00:19-04:00,2019-07-01 00:03:40-04:00,-86.16492,39.77516,0,1,7,0.396,POINT (-86.16492 39.77516)
1015756,1847fe5c4f6a60177b2249518811a31f1616a03da405c1...,233c5f053e434c87fd3fe721b6b44117d4fb27397f09ab...,2019-07-01 00:02:33-04:00,2019-07-01 00:10:24-04:00,-86.15260,39.76879,0,1,7,0.473,POINT (-86.15260 39.76879)
1015757,0cc4befe4ea796cc3958e4e082f928ebde218d66b314c7...,a9110f1d9b1df5975bc9e39d6fbd8757bea15023464a6e...,2019-07-01 00:03:04-04:00,2019-07-01 00:47:50-04:00,-86.15520,39.77409,0,1,7,2.687,POINT (-86.15520 39.77409)
1015758,58172117412704ebd442eebb71a8f1452b6d5abe7dbb99...,0b6c1ea28a1545a8165d7500e9f3f0a4b7aa8ea23f4542...,2019-07-01 00:03:44-04:00,2019-07-01 00:16:25-04:00,-86.16683,39.76567,0,1,7,0.994,POINT (-86.16683 39.76567)
1015759,ff41bcabc7fc7b8d1101f2b4a7d3e73f3ef36c6c7bf3ee...,14b4f857e27f20427a54b1b1c7d496cd15b84dff84d4ad...,2019-07-01 00:03:46-04:00,2019-07-01 00:08:24-04:00,-86.16811,39.76711,0,1,7,0.000,POINT (-86.16811 39.76711)


In [46]:
# spatial join: match each start/end point to a census tract
# The original notebook used op='intersects'. Newer GeoPandas uses predicate='intersects'.
try:
    start_within_polygon = gpd.sjoin(tract, start_point, op='intersects')
    end_within_polygon = gpd.sjoin(tract, end_point, op='intersects')
except TypeError:
    start_within_polygon = gpd.sjoin(tract, start_point, how='inner', predicate='intersects')
    end_within_polygon = gpd.sjoin(tract, end_point, how='inner', predicate='intersects')

In [47]:
# merge origin and destination information
O_data = start_within_polygon.drop('geometry', axis=1)
D_data = end_within_polygon.drop('geometry', axis=1)

OD_data = pd.merge(O_data, D_data, on='trip_id')
OD_data = OD_data[['tract_name_x','tract_name_y','trip_id','scooter_id_x','start_time_x','end_time_x','start_lon','start_lat','end_lon','end_lat','hour_x','day_x','month_x','distance_miles_x']]
OD_data.columns = ['tract_name_x','tract_name_y','trip_id','scooter_id','start_time','end_time','start_lon','start_lat','end_lon','end_lat','hour','day','month','distance_miles']

# Save with the default index, the same as the original notebook's to_csv behavior.
OD_data.to_csv('data/OD_data_reproduced.csv')
OD_data.head()

,tract_name_x,tract_name_y,trip_id,scooter_id,start_time,end_time,start_lon,start_lat,end_lon,end_lat,hour,day,month,distance_miles
0,340108,356200,f301ff9590d1df1bd40605b0d0c55bb966c9f7d859079a...,bf5f089a927e1fc7ac7c6b35f58edd5221e46fe7501bd8...,2019-07-09 13:36:17-04:00,2019-07-09 13:48:51-04:00,-86.27414,39.82095,-86.15330,39.76622,13,9,7,0.985
1,340300,340300,785eda559e043e1d716bce6000251e51a9fb82594a9782...,e0a075f1ac18f86d2e13f51fbc43a64ce91b87e7ff1e06...,2019-07-19 13:54:48-04:00,2019-07-19 14:20:02-04:00,-86.25494,39.81260,-86.25453,39.81271,13,19,7,0.026
2,340300,340400,353f8eaa74cc737ae9efac89b1995d262bb7a69abdecd2...,e0a075f1ac18f86d2e13f51fbc43a64ce91b87e7ff1e06...,2019-07-19 14:54:06-04:00,2019-07-19 15:02:26-04:00,-86.25467,39.81262,-86.23989,39.81628,14,19,7,1.188
3,340300,340300,fdf5f1d6ec96588379d40b19e8d61673cbf773e5c00edc...,9415c991fe62b9131f608608f8dcc45e9bdd3e602a9858...,2019-07-05 19:53:37-04:00,2019-07-05 21:21:04-04:00,-86.25489,39.81285,-86.25466,39.81265,19,5,7,3.437
4,340300,340300,5b5e175e3b0efd0149123cb5dc46a1962ff3822057bcf2...,fce614bfd81fe70344e010d11fb83897d0113bdb7facde...,2019-07-05 19:59:09-04:00,2019-07-05 21:01:30-04:00,-86.25459,39.81291,-86.25469,39.81267,19,5,7,4.749


## 3. Minimum fleet size calculation

The basic idea is to connect trips into trip chains. If the next trip starts after enough time and its start point is close enough to the previous trip's end point, then these two trips can be served by the same scooter.

The default thresholds below correspond to the `240 s / 300 m` scenario in Table 4.

In [48]:
# distance function
# The original notebook used transbigdata.getdistance. Here I keep a simple haversine function,
# so the notebook can run without installing transbigdata.
def getdistance(lon1, lat1, lon2, lat2):
    R = 6371000
    lon1, lat1, lon2, lat2 = map(math.radians, [lon1, lat1, lon2, lat2])
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = math.sin(dlat / 2) ** 2 + math.cos(lat1) * math.cos(lat2) * math.sin(dlon / 2) ** 2
    distance = 2 * R * math.asin(math.sqrt(a))
    return distance

In [49]:
# read OD data generated above
trip_data = pd.read_csv('data/OD_data_reproduced.csv')
trip_data = trip_data.drop(['Unnamed: 0'], axis=1)
trip_data['start_time'] = pd.to_datetime(trip_data['start_time'])
trip_data['end_time'] = pd.to_datetime(trip_data['end_time'])
trip_data['year'] = pd.to_datetime(trip_data['start_time']).dt.year
trip_data['month'] = pd.to_datetime(trip_data['start_time']).dt.month
trip_data['day'] = pd.to_datetime(trip_data['start_time']).dt.day
trip_data['week_day'] = pd.to_datetime(trip_data['start_time']).dt.weekday
trip_data.head()

,tract_name_x,tract_name_y,trip_id,scooter_id,start_time,end_time,start_lon,start_lat,end_lon,end_lat,hour,day,month,distance_miles,year,week_day
0,340108,356200,f301ff9590d1df1bd40605b0d0c55bb966c9f7d859079a...,bf5f089a927e1fc7ac7c6b35f58edd5221e46fe7501bd8...,2019-07-09 13:36:17-04:00,2019-07-09 13:48:51-04:00,-86.27414,39.82095,-86.15330,39.76622,13,9,7,0.985,2019,1
1,340300,340300,785eda559e043e1d716bce6000251e51a9fb82594a9782...,e0a075f1ac18f86d2e13f51fbc43a64ce91b87e7ff1e06...,2019-07-19 13:54:48-04:00,2019-07-19 14:20:02-04:00,-86.25494,39.81260,-86.25453,39.81271,13,19,7,0.026,2019,4
2,340300,340400,353f8eaa74cc737ae9efac89b1995d262bb7a69abdecd2...,e0a075f1ac18f86d2e13f51fbc43a64ce91b87e7ff1e06...,2019-07-19 14:54:06-04:00,2019-07-19 15:02:26-04:00,-86.25467,39.81262,-86.23989,39.81628,14,19,7,1.188,2019,4
3,340300,340300,fdf5f1d6ec96588379d40b19e8d61673cbf773e5c00edc...,9415c991fe62b9131f608608f8dcc45e9bdd3e602a9858...,2019-07-05 19:53:37-04:00,2019-07-05 21:21:04-04:00,-86.25489,39.81285,-86.25466,39.81265,19,5,7,3.437,2019,4
4,340300,340300,5b5e175e3b0efd0149123cb5dc46a1962ff3822057bcf2...,fce614bfd81fe70344e010d11fb83897d0113bdb7facde...,2019-07-05 19:59:09-04:00,2019-07-05 21:01:30-04:00,-86.25459,39.81291,-86.25469,39.81267,19,5,7,4.749,2019,4


In [50]:
# thresholds
time_threshold = 240       # seconds
distance_threshold = 300   # meters

In [51]:
# calculate the minimum fleet size for one day
# Here I use July 1, 2019 as an example.
day_data = trip_data[(trip_data['year'] == 2019) & (trip_data['month'] == 7) & (trip_data['day'] == 1)]
day_data = day_data.sort_values(by=['start_time'])

time_1 = time.time()

teams = []
total_walking_distance = 0
trip_count = 0

for i, row in day_data.iterrows():
    trip_id = row['trip_id']
    start_lat = row['start_lat']
    start_lon = row['start_lon']
    end_lat = row['end_lat']
    end_lon = row['end_lon']
    start_time = row['start_time']
    end_time = row['end_time']
    scooter_id = row['scooter_id']
    distance = row['distance_miles'] * 1609.344

    added = False
    for team in teams:
        time_diff = (start_time - team['end_time']).total_seconds()
        trip_distance = getdistance(team['end_lon'], team['end_lat'], start_lon, start_lat)

        if time_diff >= time_threshold and trip_distance <= distance_threshold:
            team['end_time'] = max(team['end_time'], end_time)
            team['end_lat'] = end_lat
            team['end_lon'] = end_lon
            team['trips'].append(trip_id)
            team['scooters'].add(scooter_id)
            team['distance'] += distance
            total_walking_distance += trip_distance
            trip_count += 1
            added = True
            break

    if not added:
        teams.append({
            'end_time': end_time,
            'end_lat': end_lat,
            'end_lon': end_lon,
            'trips': [trip_id],
            'scooters': {scooter_id},
            'distance': distance
        })

time_2 = time.time()

actual_fleet_size = len(day_data['scooter_id'].drop_duplicates())
minimum_fleet_size = len(teams)
average_walking_distance = total_walking_distance / trip_count if trip_count > 0 else 0

print('Trip amount:', len(day_data))
print('Actual fleet size:', actual_fleet_size)
print('Minimum fleet size:', minimum_fleet_size)
print('Fleet reduction rate:', (actual_fleet_size - minimum_fleet_size) / actual_fleet_size)
print('Average walking distance:', average_walking_distance)
print('Running time:', time_2 - time_1)

Trip amount: 8353
Actual fleet size: 1913
Minimum fleet size: 596
Fleet reduction rate: 0.6884474647151072
Average walking distance: 184.50161828824258
Running time: 3.5341637134552


## 4. Two-week calculation

The following cell repeats the same calculation for July 1 to July 14, 2019.

In [52]:
results = []

for j in range(1, 15):
    day_data = trip_data[(trip_data['year'] == 2019) & (trip_data['month'] == 7) & (trip_data['day'] == j)]
    day_data = day_data.sort_values(by=['start_time'])

    teams = []
    total_walking_distance = 0
    trip_count = 0

    for i, row in day_data.iterrows():
        trip_id = row['trip_id']
        start_lat = row['start_lat']
        start_lon = row['start_lon']
        end_lat = row['end_lat']
        end_lon = row['end_lon']
        start_time = row['start_time']
        end_time = row['end_time']
        scooter_id = row['scooter_id']
        distance = row['distance_miles'] * 1609.344

        added = False
        for team in teams:
            time_diff = (start_time - team['end_time']).total_seconds()
            trip_distance = getdistance(team['end_lon'], team['end_lat'], start_lon, start_lat)

            if time_diff >= time_threshold and trip_distance <= distance_threshold:
                team['end_time'] = max(team['end_time'], end_time)
                team['end_lat'] = end_lat
                team['end_lon'] = end_lon
                team['trips'].append(trip_id)
                team['scooters'].add(scooter_id)
                team['distance'] += distance
                total_walking_distance += trip_distance
                trip_count += 1
                added = True
                break

        if not added:
            teams.append({
                'end_time': end_time,
                'end_lat': end_lat,
                'end_lon': end_lon,
                'trips': [trip_id],
                'scooters': {scooter_id},
                'distance': distance
            })

    actual_fleet_size = len(day_data['scooter_id'].drop_duplicates())
    minimum_fleet_size = len(teams)
    average_walking_distance = total_walking_distance / trip_count if trip_count > 0 else 0

    results.append({
        'day': j,
        'week_day': day_data['week_day'].iloc[0],
        'Trip amount': len(day_data),
        'Actual fleet size': actual_fleet_size,
        'Minimum fleet size': minimum_fleet_size,
        'Fleet reduction rate': (actual_fleet_size - minimum_fleet_size) / actual_fleet_size,
        'Average walking distance': average_walking_distance
    })

results = pd.DataFrame(results)
results

,day,week_day,Trip amount,Actual fleet size,Minimum fleet size,Fleet reduction rate,Average walking distance
0,1,0,8353,1913,596,0.688447,184.501618
1,2,1,8308,1959,667,0.659520,183.591759
2,3,2,8046,2019,594,0.705795,184.759728
3,4,3,14135,2519,1075,0.573243,185.295489
4,5,4,9426,1988,749,0.623239,181.562666
5,6,5,9246,2087,726,0.652132,181.150685
6,7,6,5145,1802,530,0.705882,181.150186
7,8,0,3526,1513,481,0.682089,183.434036
8,9,1,4043,1616,483,0.701114,184.207015
9,10,2,3993,1572,455,0.710560,184.598482


In [53]:
# average results for the two-week period
results[['Trip amount','Actual fleet size','Minimum fleet size','Fleet reduction rate','Average walking distance']].mean()

Trip amount                 7501.928571
Actual fleet size           1913.500000
Minimum fleet size           656.142857
Fleet reduction rate           0.661792
Average walking distance     183.039428
dtype: float64

Note: this notebook is intended as a simplified reproducibility version of the original notebook. The generated `OD_data_reproduced.csv` follows the same logic as the original csv document. Small numerical differences in the fleet size calculation may appear because of different distance calculation functions or package versions.